# Lab 3: Clean Multiple CSV Files

**Week 3 · Data Engineering Course**

---

## Objective

You have three raw quarterly sales files. Each one has data quality problems. Your job is to:

1. Read all three files
2. Detect and fix every issue type
3. Merge them into one combined DataFrame
4. Write the clean combined file to `data/clean/sales_2023.csv`

**Expected output:** 73 clean rows with consistent types, no nulls in key columns, and a uniform date format.

---

## Setup

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

RAW = Path('../data/raw')
CLEAN = Path('../data/clean')
CLEAN.mkdir(parents=True, exist_ok=True)

# Verify the raw files exist
for f in ['sales_q1.csv', 'sales_q2.csv', 'sales_q3.csv']:
    path = RAW / f
    assert path.exists(), f'Missing: {path}'
    print(f'Found: {f}')

print('\nSetup complete.')

---

## Step 1 — Read and Inspect Each File

Before cleaning anything, understand what you have.

In [ ]:
q1 = pd.read_csv(RAW / 'sales_q1.csv')
q2 = pd.read_csv(RAW / 'sales_q2.csv')
q3 = pd.read_csv(RAW / 'sales_q3.csv')

for name, df in [('Q1', q1), ('Q2', q2), ('Q3', q3)]:
    print(f'--- {name} ---')
    print(f'  Shape: {df.shape}')
    print(f'  Nulls:\n{df.isnull().sum().to_string()}\n')

In [ ]:
# Spot-check categories and date formats before cleaning
for name, df in [('Q1', q1), ('Q2', q2), ('Q3', q3)]:
    print(f'{name} categories: {sorted(df["category"].unique())}')
    print(f'{name} date sample: {df["date"].head(3).tolist()}\n')

---

## Step 2 — Build Cleaning Functions

Write one function per issue type. This makes each step easy to test and reuse.

In [ ]:
def parse_date(s):
    '''Parse a date string in any of the known formats; return YYYY-MM-DD or None.'''
    for fmt in ['%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y', '%b %d %Y']:
        try:
            return datetime.strptime(str(s).strip(), fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return None

# Quick tests
assert parse_date('2023-01-05') == '2023-01-05'
assert parse_date('2023/07/14') == '2023-07-14'
assert parse_date('15/01/2023') == '2023-01-15'
assert parse_date('Jan 20 2023') == '2023-01-20'
assert parse_date('bad') is None
print('parse_date: all assertions passed')

In [ ]:
CATEGORY_TYPOS = {
    'Electrnics': 'Electronics',
    'Home & Kithen': 'Home & Kitchen'
}

def clean_category(series):
    '''Normalise casing and fix known typos.'''
    return (
        series
        .str.strip()
        .str.title()
        .replace(CATEGORY_TYPOS)
    )

def clean_price(series):
    '''Remove $ and convert to float.'''
    return pd.to_numeric(
        series.astype(str).str.replace('$', '', regex=False).str.strip(),
        errors='coerce'
    )

def fix_quantity(series):
    '''Convert to numeric, then to int.'''
    numeric = pd.to_numeric(series, errors='coerce')
    return numeric.astype('Int64')  # nullable integer type

---

## Step 3 — Clean Each File

Work on copies. Fix one issue at a time and print a count so you can verify each step.

In [ ]:
def clean_df(df, source_name):
    '''Apply all cleaning steps to a quarterly DataFrame. Returns clean copy.'''
    df = df.copy()
    print(f'\n=== Cleaning {source_name} ({len(df)} rows) ===')

    # 1 — Remove duplicate order_ids
    before = len(df)
    df = df.drop_duplicates(subset=['order_id'])
    print(f'  Duplicates removed: {before - len(df)}')

    # 2 — Strip whitespace from all string columns
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].str.strip()

    # 3 — Standardise category
    before_cats = df['category'].unique().tolist()
    df['category'] = clean_category(df['category'])
    after_cats = df['category'].unique().tolist()
    new_cats = set(before_cats) - set(after_cats)
    print(f'  Categories normalised. Fixed: {new_cats if new_cats else "none"}')

    # 4 — Parse dates
    df['date'] = df['date'].apply(parse_date)
    bad_dates = df['date'].isnull().sum()
    print(f'  Unparseable dates: {bad_dates}')

    # 5 — Clean unit_price
    df['unit_price'] = clean_price(df['unit_price'])
    print(f'  Null unit_prices after clean: {df["unit_price"].isnull().sum()}')

    # 6 — Fix quantity type
    df['quantity'] = fix_quantity(df['quantity'])

    # 7 — Recalculate missing totals
    df['total'] = pd.to_numeric(df['total'], errors='coerce')
    missing = df['total'].isnull().sum()
    df.loc[df['total'].isnull(), 'total'] = (
        df.loc[df['total'].isnull(), 'quantity'] *
        df.loc[df['total'].isnull(), 'unit_price']
    )
    print(f'  Missing totals recalculated: {missing}')

    # 8 — Drop rows with invalid quantity
    before = len(df)
    df = df[df['quantity'] > 0]
    print(f'  Rows dropped (quantity <= 0): {before - len(df)}')

    print(f'  Clean rows: {len(df)}')
    return df

q1_clean = clean_df(q1, 'Q1')
q2_clean = clean_df(q2, 'Q2')
q3_clean = clean_df(q3, 'Q3')

---

## Step 4 — Verify Each Cleaned File

Before merging, confirm each quarter looks correct.

In [ ]:
for name, df in [('Q1', q1_clean), ('Q2', q2_clean), ('Q3', q3_clean)]:
    print(f'--- {name} ---')
    print(f'  Rows: {len(df)}')
    print(f'  Nulls in key columns: {df[["date","category","unit_price","quantity","total"]].isnull().sum().to_dict()}')
    print(f'  Categories: {sorted(df["category"].unique())}')
    print(f'  date dtype: {df["date"].dtype}')
    print(f'  quantity dtype: {df["quantity"].dtype}\n')

---

## Step 5 — Merge the Three Files

In [ ]:
combined = pd.concat([q1_clean, q2_clean, q3_clean], ignore_index=True)

print(f'Combined shape: {combined.shape}')
print(f'Expected: (73, {len(combined.columns)})')
print()
print('Date range:')
print(f'  Earliest: {combined["date"].min()}')
print(f'  Latest:   {combined["date"].max()}')
print()
print('Category distribution:')
print(combined['category'].value_counts())

In [ ]:
# Final null check across the whole dataset
print('Nulls in combined file:')
print(combined.isnull().sum())
print()
print(combined.dtypes)

---

## Step 6 — Write the Clean File

In [ ]:
output_path = CLEAN / 'sales_2023.csv'
combined.to_csv(output_path, index=False)
print(f'Saved: {output_path}')

# Read it back to confirm it round-trips correctly
final = pd.read_csv(output_path)
print(f'Re-read: {final.shape}')
print(final.head(3))

---

## Step 7 — Summary Analysis

Now that the data is clean, answer three business questions.

In [ ]:
# Q1: Total revenue by category
revenue = (
    final.groupby('category')['total']
    .sum()
    .sort_values(ascending=False)
    .round(2)
)
print('Revenue by category:')
print(revenue)
print()

In [ ]:
# Q2: Top 5 products by number of orders
top_products = (
    final.groupby('product')['order_id']
    .count()
    .sort_values(ascending=False)
    .head(5)
    .rename('orders')
)
print('Top 5 products:')
print(top_products)
print()

In [ ]:
# Q3: Revenue per sales rep
rep_revenue = (
    final.dropna(subset=['sales_rep'])
    .groupby('sales_rep')['total']
    .agg(orders='count', revenue='sum')
    .sort_values('revenue', ascending=False)
    .round(2)
)
print('Revenue per sales rep:')
print(rep_revenue)

---

## Checklist Before Submitting

- [ ] `data/clean/sales_2023.csv` exists and has exactly 73 rows
- [ ] No nulls in `date`, `category`, `quantity`, `unit_price`, or `total`
- [ ] All dates are in `YYYY-MM-DD` format
- [ ] All categories are title case and free of typos
- [ ] `quantity` is integer type
- [ ] `unit_price` and `total` are float type
- [ ] All three business questions in Step 7 are answered
- [ ] Notebook runs top to bottom with no errors (`Kernel > Restart & Run All`)
- [ ] Pushed to GitHub

---

## Stretch Challenge

Add a fourth cleaning step: recalculate any **wrong** totals in Q2 where `total != quantity * unit_price` (rows 2006 and 2020). Use a boolean mask to find rows where the difference is greater than 0.01, then overwrite `total` with the correct value.